# Computational Physics: A Review

This notebook provides a review of fundamental concepts in computational physics, based on the Fortran and Python code found in this directory. Each section will cover a specific topic, explaining the underlying theory, the Fortran implementation, and any accompanying Python analysis or visualization.

## 1. Series Summations and Floating-Point Precision

One of the first things to consider in numerical computation is the limitation of floating-point arithmetic. The way numbers are stored in a computer means that there is finite precision, which can lead to rounding errors. These errors can accumulate, especially when dealing with a large number of operations, such as summing a long series.

A classic example is the summation of a harmonic series: S = Σ (1/n) from n=1 to N. When N is large, we are adding very small numbers to a growing sum. If the sum becomes large, the smaller terms might be smaller than the precision of the sum, and their contribution is lost. This is called **loss of significance**.

To demonstrate this, we can sum the series in two ways: 'up' (from n=1 to N) and 'down' (from n=N to 1). When summing up, we add smaller terms to a progressively larger sum, which can lead to a loss of precision. When summing down, we add smaller terms to a smaller sum, which is more accurate. The difference between `sum_up` and `sum_down` reveals the effect of floating-point error.

### Fortran Implementation

The file `assgn_1_1.f90` calculates the sum of the harmonic series up and down for different values of N.

In [ ]:
```fortran
program summation
    implicit none
    
    integer, parameter :: start_N = 1000
    integer, parameter :: end_N = 1000000
    integer, parameter :: step = 1000
    integer :: N
    real :: sum_up, sum_down
    
    open(unit=10, file='output.txt', status='replace')
    
    do N = start_N, end_N, step
        sum_up = calculate_sum_up(N)
        sum_down = calculate_sum_down(N)
        write(10, '(I10, 2E16.6)') N, sum_up, sum_down
    end do
    
    close(unit=10)
    
contains

    real function calculate_sum_up(N)
        implicit none
        integer, intent(in) :: N
        integer :: n
        
        calculate_sum_up = 0.0
        do n = 1, N
            calculate_sum_up = calculate_sum_up + 1.0_real / real(n, kind=selected_real_kind(6, 37))
        end do
    end function calculate_sum_up

    real function calculate_sum_down(N)
        implicit none
        integer, intent(in) :: N
        integer :: n
        
        calculate_sum_down = 0.0
        do n = N, 1, -1
            calculate_sum_down = calculate_sum_down + 1.0_real / real(n, kind=selected_real_kind(6, 37))
        end do
    end function calculate_sum_down

end program summation
```

The file `series_1.f90` does a similar calculation, but for both single and double precision, and calculates the relative difference between the up and down summations.

In [ ]:
```fortran
program series_calculation
    implicit none
    integer, parameter :: start_N = 1000, end_N = 1000000, step = 1000
    integer :: N, i
    real(kind=4) :: S_up, S_down, diff
    real(kind=4), dimension((end_N - start_N) / step + 1) :: result_single
    real(kind=8) :: S_up_d, S_down_d, diff_d
    real(kind=8), dimension((end_N - start_N) / step + 1) :: result_double
    open(unit=10, file='output_single_precision.txt')
    open(unit=20, file='output_double_precision.txt')

    ! Single-precision calculation
    do N = start_N, end_N, step
        S_up = 0.0
        S_down = 0.0
        do i = 1, N
            S_up = S_up + 1.0 / real(i, kind=4)
        end do
        do i = N, 1, -1
            S_down = S_down + 1.0 / real(i, kind=4)
        end do
        diff = abs(S_up - S_down) / (S_up + S_down)
        result_single((N - start_N) / step + 1) = diff
        write(10, *) N, diff
    end do

    ! Double-precision calculation
    do N = start_N, end_N, step
        S_up_d = 0.0d0
        S_down_d = 0.0d0
        do i = 1, N
            S_up_d = S_up_d + 1.0d0 / real(i, kind=8)
        end do
        do i = N, 1, -1
            S_down_d = S_down_d + 1.0d0 / real(i, kind=8)
        end do
        diff_d = abs(S_up_d - S_down_d) / (S_up_d + S_down_d)
        result_double((N - start_N) / step + 1) = diff_d
        write(20, *) N, diff_d
    end do

    close(10)
    close(20)


end program series_calculation
```

### Python Visualization

The Python script `plot_sumseries.py` can be used to visualize the results from `assgn_1_1.f90`. It reads the `output.txt` file and plots the difference between the 'up' and 'down' sums as a function of N.

In [ ]:
```python
import numpy as np
import matplotlib.pyplot as plt

# Read data from output.txt
data = np.loadtxt('output.txt')

# Extracting columns
N_values = data[:, 0]
sum_up_values = data[:, 1]
sum_down_values = data[:, 2]
sum_diff_values = np.abs(sum_up_values - sum_down_values)

# Plotting with log scales
plt.figure(figsize=(10, 6))
plt.plot(N_values, sum_diff_values, label='sum_diff')
plt.xscale('log')
plt.yscale('log')
plt.title('Sum Difference vs N')
plt.xlabel('N (log scale)')
plt.ylabel('Sum Difference (log scale)')
plt.legend()
plt.grid(True)
plt.show()
```